In [ ]:
import pandas as pd
from scipy import stats
import numpy as np
from scipy.stats import false_discovery_control
import os
from plot_tools import *
from scipy.stats import chi2
from snp_analysis_tools_sherlock import *
from scipy.stats import binom
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot
from glob import glob

hv.extension('bokeh')

In [ ]:
df = pd.read_csv('~/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/101346/AA-AF-mBHI_parentboth_info.csv')
df_both = get_selection_stuff_both(101346,'AA-AF-mBHI')
species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
all_dfs = []
for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
     #   try:
        if os.path.exists(f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/{sp}/{ino}_parentboth_info.csv'):
            df,_=get_selection_stuff_both(sp,ino)#.reset_index()
       # except:
        #    continue
            df['species_id']=sp
            df['inoculumn']=ino
            df['species-ino']= sp+'-' + ino
            all_dfs.append(df.reset_index())
            
all_dfs=pd.concat(all_dfs)
print(len(all_dfs['species_id'].unique()))
all_dfs = all_dfs.loc[(all_dfs['passage1']==0)*((all_dfs['passage2']==7)),:]
all_dfs['sel_coeff']=all_dfs['sel_med']
bad_inos = ['100146-AA-AF-mBHI',
 '100146-AE-AF-mBHI',
 '102478-AA-AF-mBHI',
 '102478-AE-AF-mBHI',
 '100196-AA-AF-mBHI',
 '100196-AE-AF-mBHI',
 '100196-AA-AE-mGAM',
 '100196-AE-AF-mGAM',
 '102544-AA-AF-mBHI',
 '102544-AE-AF-mBHI',
 '101349-AA-AF-mBHI',
 '101349-AE-AF-mBHI',
 '102506-AA-AE-mBHI',
 '102506-AE-AF-mBHI',
 '102506-AA-AF-mBHI',
 '102506-AE-AF-mBHI',
 '102506-AA-AE-mGAM',
 '102506-AE-AF-mGAM',
 '101346-AA-AE-mBHI',
 '101346-AE-AF-mBHI',
 '101346-AA-AF-mBHI',
 '101346-AE-AF-mBHI',
 '101346-AA-AE-mGAM',
 '101346-AE-AF-mGAM',
 '101346-AA-AE-mBHI',
 '101346-AA-AF-mBHI',
 '100120-AA-AF-mBHI',
 '100120-AE-AF-mBHI',
 '102327-AA-AF-mBHI',
 '102327-AE-AF-mBHI']

print(len(all_dfs))
all_dfs = all_dfs.groupby(['sample1','sample2','mesocosm','inoculumn','type_mesocosm','species-ino','species_id','media']).median(numeric_only=True).reset_index()
all_dfs = all_dfs.loc[~all_dfs['species-ino'].isin(bad_inos),:]
print(len(all_dfs))
len(all_dfs['species_id'].unique())


In [ ]:
mGAMs_max=[]
mGAMs_min=[]
mBHIs_max=[]
mBHIs_min=[]
mBHI_est=[]
mGAM_est=[]
sp_inos=[]
pvals=[]
for sp_ino in all_dfs['species-ino'].unique():
    df_sp_ino = all_dfs.loc[all_dfs['species-ino']==sp_ino,:]
    df_sp_ino=df_sp_ino.loc[~df_sp_ino['sel_med'].isna(),:]
    media = df_sp_ino['media'].unique()
    if len(media)<2:
        print(sp_ino, media)
        continue
    mGAMs = df_sp_ino.loc[df_sp_ino['media']=='mGAM','sel_med'].values
    mBHIs = df_sp_ino.loc[df_sp_ino['media']=='mBHI','sel_med'].values
    if len(mBHIs) <1 or len(mGAMs) < 1:
        print(sp_ino)
        continue
    
    mGAMs_min.append(np.min(mGAMs))
    mGAMs_max.append(np.max(mGAMs))
    mGAM_est.append(np.median(mGAMs))
 
    print(sp_ino,np.min(mGAMs)>0, np.max(mGAMs)>0, np.median(mGAMs)>0)
    

    mBHIs_min.append(np.min(mBHIs))
    mBHIs_max.append(np.max(mBHIs))
    mBHI_est.append(np.median(mBHIs))

    sp_inos.append(sp_ino)
    t_statistic, p_value = stats.ttest_ind(mGAMs, mBHIs)
    pvals.append(p_value)

In [ ]:

df = pd.DataFrame(data={'mGAM_min':mGAMs_min,'mBHI_min':mBHIs_min,
                        'mGAM_max':mGAMs_max,'mBHI_max':mBHIs_max,
                        'mBHI':mBHI_est,
                        'mGAM':mGAM_est,
                        'pvals':pvals,
                        'species-inoculumn': sp_inos})
df=df.sort_values(by='pvals')
df=df.loc[~df['pvals'].isna(),:]
qvals = np.sort(df['pvals'].values)
qvals = stats.false_discovery_control(np.sort(qvals[~np.isnan(qvals)]))
df['qvals']=qvals
df.loc[df['qvals']<.05,:]
len(df)
print(len(df))
print(len(df.loc[df['qvals']<.05,:]))

In [ ]:
df['species_id']=df['species-inoculumn'].transform(lambda x: x.split('-')[0])
print(len(df['species_id'].unique()))

In [ ]:


df = pd.DataFrame(data={'mGAM_min':mGAMs_min,'mBHI_min':mBHIs_min,
                        'mGAM_max':mGAMs_max,'mBHI_max':mBHIs_max,
                        'mBHI':mBHI_est,
                        'mGAM':mGAM_est,
                        'pvals':pvals,
                        'species-inoculumn': sp_inos})


df=df.sort_values(by='pvals')
df=df.loc[~df['pvals'].isna(),:]
qvals = np.sort(df['pvals'].values)
qvals = stats.false_discovery_control(np.sort(qvals[~np.isnan(qvals)]))
df['qvals']=qvals


df['repol']=False
df.loc[df['mGAM']<0,'repol']=True
to_repol = df.loc[df['repol'],'species-inoculumn'].unique()
df.loc[df['mGAM']<0,'mBHI']=-df.loc[df['mGAM']<0,'mBHI']
df.loc[df['mGAM']<0,'mGAM']=-df.loc[df['mGAM']<0,'mGAM']
df = df.sort_values(by='qvals')
df = df.loc[df['species-inoculumn']!='100910-AA-AE-mBHI',:]

df['species']=df['species-inoculumn'].transform(lambda x: x.split('-')[0])

#df.loc[df['qvals']<.05,'sp_plot-ino']=df.loc[df['qvals']<.05,'sp_plot-ino']+'*'
#sig = df.loc[df['qvals']<.05,'sp_plot-ino'].unique()
#df.loc[df['qvals']<.001,'sp_plot-ino']=df.loc[df['qvals']<.001,'sp_plot-ino']+'**'
#sigsig = df.loc[df['qvals']<.001,'sp_plot-ino'].unique()
df = df.sort_values(by='mGAM',ascending=False)
p = hv.Points(df, vdims=['mGAM'],kdims=['species-inoculumn','mGAM'], label='mGAM',).opts(width=700,
                                                                                                     size=5,
                                                                                                color=bokeh.palettes.Set2[8][0],
                                                                                               line_color='black',
                                                                                                 xrotation=60)





p2=hv.Points(df, vdims=['mBHI'],kdims=['species-inoculumn','mBHI'],label='mBHI',).opts(width=500,
                                                                                                    size=5,
                                                                                                   line_color='black',
                                                                                    color=bokeh.palettes.Set2[8][1],
                                                                                                              legend_position='right',
                                                                                                xrotation=60)


all_dfs['species-inoculumn']=all_dfs['species-ino']
all_dfs_p= all_dfs.loc[all_dfs['species-inoculumn'].isin(df['species-inoculumn'].values),:]

all_dfs_p.loc[all_dfs_p['species-inoculumn'].isin(to_repol),'sel_coeff']= -all_dfs_p.loc[all_dfs_p['species-inoculumn'].isin(to_repol),'sel_coeff']

all_dfs_p['species']=all_dfs_p['species-inoculumn'].transform(lambda x: x.split('-')[0])
#all_dfs_p['sp_plot']=all_dfs_p['species'].transform(lambda x: df_metadata.loc[x,'species_plot'])
#all_dfs_p['sp_plot'] = all_dfs_p['sp_plot'].transform(lambda x: x.split(' ')[0][0] +' '+ x.split(' ')[1].split('_')[0][:5])
#all_dfs_p['sp_plot-ino'] = all_dfs_p['sp_plot']+' in '+  all_dfs_p['species-inoculumn'].transform(lambda x: x[7:])

#all_dfs_p.loc[all_dfs_p['sp_plot-ino'].isin(sig),'sp_plot-ino']= all_dfs_p.loc[all_dfs_p['sp_plot-ino'].isin(sig),'species-inoculumn']+'*'
#all_dfs_p.loc[all_dfs_p['sp_plot-ino'].isin(sigsig),'sp_plot-ino']=all_dfs_p.loc[all_dfs_p['sp_plot-ino'].isin(sigsig),'species-inoculumn'] +'**'
p3 = hv.Points(all_dfs_p.loc[all_dfs_p['media']=='mGAM',:], vdims=['sel_coeff'],kdims=['species-inoculumn','sel_coeff']).opts(width=500,
                                                                                                    size=5,
                                                                                    color=bokeh.palettes.Set2[8][0],
                                                                                          alpha=.5,
                                                                                                xrotation=60)

p4 = hv.Points(all_dfs_p.loc[all_dfs_p['media']=='mBHI',:], vdims=['sel_coeff'],kdims=['species-inoculumn','sel_coeff']).opts(width=500,
                                                                                                    size=5,
                                                                                    color=bokeh.palettes.Set2[8][1],
                                                                                          alpha=.5,
                                                                                                xrotation=60)
p=hv.render(p*p2*p3*p4*p*p2)
p.yaxis.axis_label='Sel Coeff'
p.xaxis.axis_label='Competition'
p.ray(x=0,y=0,angle=0,color='black')
bokeh.io.show(p)

In [ ]:


df = pd.DataFrame(data={'mGAM_min':mGAMs_min,'mBHI_min':mBHIs_min,
                        'mGAM_max':mGAMs_max,'mBHI_max':mBHIs_max,
                        'mBHI':mBHI_est,
                        'mGAM':mGAM_est,
                        'pvals':pvals,
                        'species-inoculumn': sp_inos})
df=df.sort_values(by='pvals')
df=df.loc[~df['pvals'].isna(),:]
qvals = np.sort(df['pvals'].values)
qvals = stats.false_discovery_control(np.sort(qvals[~np.isnan(qvals)]))
df['qvals']=qvals
df.loc[df['qvals']<.05,:]
df['repol']=False
df.loc[df['mBHI']<0,'repol']=True
to_repol = df.loc[df['repol'],'species-inoculumn'].unique()
df.loc[df['mBHI']<0,'mGAM']=-df.loc[df['mBHI']<0,'mGAM']
df.loc[df['mBHI']<0,'mBHI']=-df.loc[df['mBHI']<0,'mBHI']
df = df.sort_values(by='qvals')
df = df.loc[df['species-inoculumn']!='100910-AA-AE-mBHI',:]

df['species']=df['species-inoculumn'].transform(lambda x: x.split('-')[0])

df = df.sort_values(by='mBHI',ascending=False)
p = hv.Points(df, vdims=['mGAM'],kdims=['species-inoculumn','mGAM'], label='mGAM',).opts(width=700,
                                                                                                     size=5,
                                                                                                color=bokeh.palettes.Set2[8][0],
                                                                                               line_color='black',
                                                                                                 xrotation=60)





p2=hv.Points(df, vdims=['mBHI'],kdims=['species-inoculumn','mBHI'],label='mBHI',).opts(width=500,
                                                                                                    size=5,
                                                                                                   line_color='black',
                                                                                    color=bokeh.palettes.Set2[8][1],
                                                                                                              legend_position='right',
                                                                                                xrotation=60)


all_dfs['species-inoculumn']=all_dfs['species-ino']
all_dfs_p= all_dfs.loc[all_dfs['species-inoculumn'].isin(df['species-inoculumn'].values),:]
all_dfs_p.loc[all_dfs_p['species-inoculumn'].isin(to_repol),'sel_coeff']= -all_dfs_p.loc[all_dfs_p['species-inoculumn'].isin(to_repol),'sel_coeff']

all_dfs_p['species']=all_dfs_p['species-inoculumn'].transform(lambda x: x.split('-')[0])

p3 = hv.Points(all_dfs_p.loc[all_dfs_p['media']=='mGAM',:], vdims=['sel_coeff'],kdims=['species-inoculumn','sel_coeff'],).opts(width=500,
                                                                                                    size=5,
                                                                                    color=bokeh.palettes.Set2[8][0],
                                                                                          alpha=.5,
                                                                                                xrotation=60)

p4 = hv.Points(all_dfs_p.loc[all_dfs_p['media']=='mBHI',:], vdims=['sel_coeff'],kdims=['species-inoculumn','sel_coeff']).opts(width=500,
                                                                                                    size=5,
                                                                                    color=bokeh.palettes.Set2[8][1],
                                                                                          alpha=.5,
                                                                                                xrotation=60)

dfsig = df.loc[df['qvals']<.05,:].copy()
dfsig  = dfsig.loc[dfsig['qvals']>.001,:]
dfsig['sel_coeff']= -1.5
dfsig['media'] = 'p<.05'
p5 = hv.Points(dfsig, vdims=['sel_coeff'],kdims=['species-inoculumn','sel_coeff'], label = 'p<.05').opts(width=500,marker='*',color = 'black',
                                                                                                    size=5,
                                                                                )

dfsigsig = df.loc[df['qvals']<.001,:].copy()
dfsigsig['sel_coeff']= -1.5
dfsigsig['media'] = 'p<.05'
p6 = hv.Points(dfsigsig, vdims=['sel_coeff'],kdims=['species-inoculumn','sel_coeff'],  label = 'p<.001').opts(width=500,marker='+',color = 'black',
                                                                                                    size=5,
                                                                                )
p=hv.render(p*p2*p3*p4*p2*p5*p6)
p.yaxis.axis_label='Sel Coeff'
p.xaxis.axis_label='Competition'
p.ray(x=0,y=0,angle=0,color='black')
p.output_backend = "svg"

test_name='bloh'
bokeh.io.export_svgs(p, filename = test_name + '.svg')
export_plot_pdf(p,'media_stuff')
bokeh.io.show(p)

In [ ]:
df.loc[df['mGAM']<0,:].sort_values(by='qvals')